# MovieLens movies: Bronze Catalog table to silver

This AWS Glue notebook reads `movielens.movies` as a `DynamicFrame`, converts it to a Spark `DataFrame` for cleansing, converts it back to a `DynamicFrame`, and writes Parquet while registering `movielens.movies_silver`.

## Before running

Run in an AWS Glue notebook/interactive session. The execution role needs read access to the source, write access to the silver S3 prefix, and Glue Catalog read/update permissions. Replace `SILVER_ROOT`; use an empty dedicated prefix on the first run.

In [ ]:
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types as T

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

DATABASE = "movielens"
SOURCE_TABLE = "movies"
TARGET_TABLE = "movies_silver"
SILVER_ROOT = "s3://YOUR-BUCKET/silver/movielens".rstrip("/")
TARGET_PATH = f"{SILVER_ROOT}/{TARGET_TABLE}/"

assert "YOUR-BUCKET" not in SILVER_ROOT, "Set SILVER_ROOT before running"
print(f"Source: {DATABASE}.{SOURCE_TABLE}")
print(f"Target: {DATABASE}.{TARGET_TABLE} -> {TARGET_PATH}")

## Read from the Glue Data Catalog as a DynamicFrame

In [ ]:
movies_raw_dyf = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE,
    table_name=SOURCE_TABLE,
    transformation_ctx="movies_raw_dyf",
)
print(f"Raw rows: {movies_raw_dyf.count():,}")
movies_raw_dyf.printSchema()

## DynamicFrame → DataFrame

Spark DataFrames provide the broadest set of SQL expressions, so the business transformation happens there. The resolver makes the notebook tolerant of crawler-created column-name casing.

In [ ]:
movies_raw_df = movies_raw_dyf.toDF()

def source_column(df, expected):
    matches = [name for name in df.columns if name.lower() == expected.lower()]
    if len(matches) != 1:
        raise ValueError(f"Expected one column named {expected!r}; found {matches}. Available: {df.columns}")
    return F.col(f"`{matches[0]}`")

movie_id = source_column(movies_raw_df, "movieId")
title = source_column(movies_raw_df, "title")
genres = source_column(movies_raw_df, "genres")

In [ ]:
movies_silver_df = (
    movies_raw_df
    .select(
        movie_id.cast(T.LongType()).alias("movie_id"),
        F.trim(title).alias("title"),
        F.trim(genres).alias("genres_raw"),
    )
    .withColumn("release_year", F.regexp_extract("title", r"\((\d{4})\)$", 1).cast("int"))
    .withColumn("genres", F.when(F.col("genres_raw") == "(no genres listed)", F.array().cast("array<string>")).otherwise(F.split("genres_raw", r"\|")))
    .withColumn("ingested_at_utc", F.current_timestamp())
    .filter(F.col("movie_id").isNotNull() & F.col("title").isNotNull() & (F.length("title") > 0))
    .dropDuplicates(["movie_id"])
    .select("movie_id", "title", "release_year", "genres", "genres_raw", "ingested_at_utc")
)

movies_silver_df.printSchema()
movies_silver_df.show(10, truncate=False)

## DataFrame → DynamicFrame and validation

In [ ]:
movies_silver_dyf = DynamicFrame.fromDF(movies_silver_df, glueContext, "movies_silver_dyf")
movies_silver_dyf.printSchema()

raw_count = movies_raw_dyf.count()
silver_count = movies_silver_dyf.count()
invalid_count = movies_silver_df.filter(F.col("movie_id").isNull() | F.col("title").isNull()).count()
duplicate_count = movies_silver_df.groupBy("movie_id").count().filter(F.col("count") > 1).count()
print({"raw_count": raw_count, "silver_count": silver_count, "invalid_count": invalid_count, "duplicate_movie_ids": duplicate_count})
assert invalid_count == 0
assert duplicate_count == 0

## Write Parquet and update the Catalog

The Glue sink writes data files and creates or updates `movielens.movies_silver`. Glue sinks append files; for repeatable full refreshes, clear only this dedicated target prefix through an approved lifecycle/job step before rerunning, or adopt a versioned/incremental design.

In [ ]:
sink = glueContext.getSink(
    connection_type="s3",
    path=TARGET_PATH,
    enableUpdateCatalog=True,
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=[],
    transformation_ctx="movies_silver_sink",
)
sink.setCatalogInfo(catalogDatabase=DATABASE, catalogTableName=TARGET_TABLE)
sink.setFormat("glueparquet", compression="snappy")
sink.writeFrame(movies_silver_dyf)
print(f"Published {DATABASE}.{TARGET_TABLE} at {TARGET_PATH}")

In [ ]:
published_df = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE, table_name=TARGET_TABLE, transformation_ctx="movies_published_check"
).toDF()
published_df.printSchema()
published_df.show(10, truncate=False)